In [1]:
import pandas as pd
import os

In [2]:
%run analysis_utils.py

In [3]:

from pathlib import Path

def find_files(root_folder, filename):
    """
    Search for all instances of a file inside a folder and its subfolders.

    Parameters
    ----------
    root_folder : str or Path
        Folder to search inside.
    filename : str
        Exact file name to search for, e.g. "data.h5ad".

    Returns
    -------
    list[Path]
        List of matching file paths.
    """
    root_folder = Path(root_folder)

    if not root_folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {root_folder}")

    matches = list(root_folder.rglob(filename))
    return matches


# Example usage
root = '/home/haitham/mnt/__output_v7/exp'
# root = '/home/haitham/mnt/__output_v3/exp'


vote_files = find_files(root, 'vote_cv_metrics.csv')
avg_files = find_files(root, 'avg_cv_metrics.csv')
mil_files = find_files(root, 'mil_cv_metrics.csv')

# print(f"Found {len(matches)} matching file(s):")
# for path in matches:
#     print(path)

In [4]:
model_name_map={
'hvg': 'HVG',
'pca': 'PCA',
'scgpt': 'scGPT', 
'scgpt_cancer': 'scGPT [cancer]',
'scvi':'scVI',
'scvi_donor_id':'scVI',
'scfoundation':'scFoundation',
'scimilarity':'SCimiarity',
'cellplm':'CellPLM',
'gf-6L-30M-i2048': 'GF-V1',
'gf-6L-30M-i2048_continue': 'GF-V1 [continue]',
'Geneformer-V2-104M_CLcancer': 'GF-V2 [cancer]',
'Geneformer-V2-104M': 'GF-V2',
'Geneformer-V2-104M_continue': 'GF-V2 [continue]',
'Geneformer-V2-316M': 'GF-V2-Deep',
'gf-6L-30M-i2048_finetune': 'GF-V1 [finetune]',
'Geneformer-V2-104M_finetune': 'GF-V2 [finetune]',
'hvg_seurat_4096': 'HVG' ,                        
'state_se600m_epoch16': 'STATE',                     
'scfoundation_brca_cancer_cells': 'scFoundation',         
'geneformer_V2-104M_CLcancer-i4096':'GF-V2 [cancer]',
'geneformer_V2-316M-i4096':'GF-V2-Deep'                    , 
'continue_geneformer_V1-10M-i2048_continue':'GF-V1 [continue]',     
'geneformer_V1-10M-i2048':'GF-V1'       ,
'continue_geneformer_V2-104M-i4096_continue': 'GF-V2 [continue]'  ,  
'geneformer_V2-104M-i4096': 'GF-V2'                 ,
'scgpt_cancer-i2048' :'scGPT [cancer]',                           
'scgpt_human-i2048'  : 'scGPT',                          
'cellplm_85M-20231027'  :'CellPLM',                       
'scimilarity_v1.1': 'SCimiarity'                              ,
'pca_n100'  :'PCA [100]'  ,
'pca_n50'   : 'PCA [50]',                                   
'pca_n20'                : 'PCA [20]'        ,               
'scvi'                               :'scVI'     ,      
'scconcept_corpus30m' : 'scConcept'                          ,
'nicheformer_nicheformer'                       :'Nicheformer',


}

In [5]:
experiment_name_map=dict(pre_post ='Treatment Naive vs Anti PD1',
                         brca_full_pre_post ='Treatment Naive vs Anti PD1',
                         brca_pre_post ='Treatment Naive vs Anti PD1',
                         chemo= 'Treatment Naive vs Neoadjuvant Chemo',
                         brca_full_chemo= 'Treatment Naive vs Neoadjuvant Chemo',
                         brca_chemo= 'Treatment Naive vs Neoadjuvant Chemo',
                         luad2 = 'Treatment Naive vs TKI treated',
                         luad_tki = 'Treatment Naive vs TKI treated',
                        # luad22 = 'Treatment Naive vs TKI treated',
                         luad1 = 'Early stage vs Late stage',
                        # luad11 = 'Early stage vs Late stage',
                         outcome ='T-cell exhaustion',
                         brca_full_outcome ='T-cell exhaustion',
                         brca_outcome ='T-cell exhaustion',
                         subtype= 'ER+ vs TNBC',
                         brca_full_subtype= 'ER+ vs TNBC', 
                        brca_subtype= 'ER+ vs TNBC', 
                        brca_cell_type= 'BRCA Cell Type', melanoma_response='IO Response', crc_mmr='MMRd vs MMRp')

In [6]:
import json
vote_dfs = []
for matched_file in vote_files:
    df = pd.read_csv(matched_file)
    df = df.pivot(index='Metrics', columns='fold', values='randomforest').T
    run_summary_path = matched_file.parent.parent / "run_summary.json"
    try:
        with open(run_summary_path, "r") as f:
                run_summary = json.load(f)
                model = run_summary['run_id']
                exp = Path(run_summary['config_path']).stem
                model  = model.replace(f'_{exp}', "")
    except:
       print(f'cannot load summary file. Skipping: {run_summary_path}')
       continue
        
    df['model'] = model
    df['exp'] = exp
    df['strategy'] = 'vote'
    vote_dfs.append(df)
        
    

In [7]:
vote_dfs[0]

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
fold,,,,,,,,,
fold_1,1.0,1.0,1.0,1.0,1.0,1.0,hvg_seurat_4096,luad_tki,vote
fold_2,1.0,1.0,1.0,1.0,1.0,1.0,hvg_seurat_4096,luad_tki,vote
fold_3,1.0,1.0,1.0,1.0,1.0,1.0,hvg_seurat_4096,luad_tki,vote
fold_4,1.0,1.0,1.0,1.0,1.0,1.0,hvg_seurat_4096,luad_tki,vote


In [8]:
vote_df = pd.concat(vote_dfs)
vote_df = vote_df.reset_index().drop('fold', axis=1)
# vote_df.exp = vote_df.exp.map(experiment_name_map)
vote_df

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,vote
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,vote
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,vote
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,vote
4,0.888889,0.933333,0.833333,0.828571,0.875000,0.833333,hvg_seurat_4096,crc_mmr,vote
...,...,...,...,...,...,...,...,...,...
475,0.857143,0.500000,0.875000,0.466667,0.437500,0.500000,nicheformer_nicheformer,brca_chemo,vote
476,0.416667,0.309524,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,vote
477,0.833333,0.750000,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,vote
478,0.750000,0.700000,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,vote


In [9]:
vote_df.exp.unique()

array(['luad_tki', 'crc_mmr', 'brca_pre_post', 'brca_outcome',
       'brca_chemo', 'melanoma_response', 'brca_subtype',
       'luad_cancer_stage'], dtype=object)

In [10]:
vote_df.exp.map(experiment_name_map)

0            Treatment Naive vs TKI treated
1            Treatment Naive vs TKI treated
2            Treatment Naive vs TKI treated
3            Treatment Naive vs TKI treated
4                              MMRd vs MMRp
                       ...                 
475    Treatment Naive vs Neoadjuvant Chemo
476    Treatment Naive vs Neoadjuvant Chemo
477    Treatment Naive vs Neoadjuvant Chemo
478    Treatment Naive vs Neoadjuvant Chemo
479    Treatment Naive vs Neoadjuvant Chemo
Name: exp, Length: 480, dtype: object

In [11]:
avg_dfs = []
for matched_file in avg_files:
    df = pd.read_csv(matched_file)
    df = df.pivot(index='Metrics', columns='fold', values='randomforest').T
    run_summary_path = matched_file.parent.parent / "run_summary.json"
    try:
        with open(run_summary_path, "r") as f:
                
            run_summary = json.load(f)
            model = run_summary['run_id']
            exp = Path(run_summary['config_path']).stem
            model  = model.replace(f'_{exp}', "")
    except:
           print(f'cannot load summary file. Skipping: {run_summary_path}')
           continue
        
    df['model'] = model
    df['exp'] = exp
    df['strategy'] = 'avg'
    avg_dfs.append(df)
    

In [12]:
avg_df = pd.concat(avg_dfs)
avg_df = avg_df.reset_index().drop('fold', axis=1)
# avg_df.exp = avg_df.exp.map(experiment_name_map)

avg_df

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,avg
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,avg
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,avg
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,avg
4,1.000000,1.000000,0.750000,0.733333,0.833333,0.750000,hvg_seurat_4096,crc_mmr,avg
...,...,...,...,...,...,...,...,...,...
475,0.714286,0.333333,0.875000,0.466667,0.437500,0.500000,nicheformer_nicheformer,brca_chemo,avg
476,0.666667,0.450000,0.750000,0.666667,0.666667,0.666667,nicheformer_nicheformer,brca_chemo,avg
477,1.000000,1.000000,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,avg
478,0.833333,0.750000,0.875000,0.794872,0.928571,0.750000,nicheformer_nicheformer,brca_chemo,avg


In [13]:
# avg_df[avg_df['model'].str.contains('geneformer_V1-10M')& avg_df['exp'].str.contains('brca_pre_post')]

In [14]:
mil_dfs = []
for matched_file in mil_files:
    df = pd.read_csv(matched_file)
    df = df.pivot(index='Metrics', columns='fold', values='randomforest').T
    run_summary_path = matched_file.parent.parent / "run_summary.json"
    try: 
        with open(run_summary_path, "r") as f:
                run_summary = json.load(f)
                model = run_summary['run_id']
                exp = Path(run_summary['config_path']).stem
                model  = model.replace(f'_{exp}', "")
    except:
       print(f'cannot load summary file. Skipping: {run_summary_path}')
       continue
        
    df['model'] = model
    df['exp'] = exp
    df['strategy'] = 'MIL'
    mil_dfs.append(df)

In [15]:
mil_df = pd.concat(mil_dfs)
mil_df = mil_df.reset_index().drop('fold', axis=1)
# mil_df.exp = mil_df.exp.map(experiment_name_map)

mil_df


Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,MIL
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,MIL
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,MIL
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,hvg_seurat_4096,luad_tki,MIL
4,1.000000,1.000000,0.666667,0.625000,0.800000,0.666667,hvg_seurat_4096,crc_mmr,MIL
...,...,...,...,...,...,...,...,...,...
475,0.857143,0.500000,0.875000,0.466667,0.437500,0.500000,nicheformer_nicheformer,brca_chemo,MIL
476,0.500000,0.392857,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,MIL
477,0.750000,0.700000,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,MIL
478,0.666667,0.666667,0.750000,0.428571,0.375000,0.500000,nicheformer_nicheformer,brca_chemo,MIL


In [16]:
calssification_metrics = pd.concat([vote_df,mil_df, avg_df ])

In [17]:
calssification_metrics.model.unique()

array(['hvg_seurat_4096', 'state_se600m_epoch16', 'scfoundation',
       'geneformer_V2-104M_CLcancer-i4096', 'geneformer_V2-316M-i4096',
       'geneformer_V1-10M-i2048', 'geneformer_V2-104M-i4096',
       'scgpt_cancer-i2048', 'scgpt_human-i2048', 'cellplm_85M-20231027',
       'scimilarity_v1.1', 'pca_n100', 'scvi', 'nicheformer_nicheformer'],
      dtype=object)

In [18]:
model_name_map['hvg_seurat_4096']

'HVG'

In [19]:
calssification_metrics.exp.value_counts()

exp
crc_mmr              210
brca_pre_post        210
brca_outcome         210
brca_chemo           210
brca_subtype         210
melanoma_response    210
luad_tki             168
luad_cancer_stage     12
Name: count, dtype: int64

In [20]:
calssification_metrics = calssification_metrics[calssification_metrics.exp!='luad_cancer_stage']

calssification_metrics['group'] =  calssification_metrics['model'].map(map_groups)
calssification_metrics['model'] =  calssification_metrics['model'].map(model_name_map)
calssification_metrics.exp = calssification_metrics.exp.map(experiment_name_map)

calssification_metrics

Metrics,AUC,AUPRC,Accuracy,F1,Precision,Recall,model,exp,strategy,group
0,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,HVG,Treatment Naive vs TKI treated,vote,Baseline
1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,HVG,Treatment Naive vs TKI treated,vote,Baseline
2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,HVG,Treatment Naive vs TKI treated,vote,Baseline
3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,HVG,Treatment Naive vs TKI treated,vote,Baseline
4,0.888889,0.933333,0.833333,0.828571,0.875000,0.833333,HVG,MMRd vs MMRp,vote,Baseline
...,...,...,...,...,...,...,...,...,...,...
475,0.714286,0.333333,0.875000,0.466667,0.437500,0.500000,Nicheformer,Treatment Naive vs Neoadjuvant Chemo,avg,Other
476,0.666667,0.450000,0.750000,0.666667,0.666667,0.666667,Nicheformer,Treatment Naive vs Neoadjuvant Chemo,avg,Other
477,1.000000,1.000000,0.750000,0.428571,0.375000,0.500000,Nicheformer,Treatment Naive vs Neoadjuvant Chemo,avg,Other
478,0.833333,0.750000,0.875000,0.794872,0.928571,0.750000,Nicheformer,Treatment Naive vs Neoadjuvant Chemo,avg,Other


In [21]:
calssification_metrics.exp.value_counts()

exp
MMRd vs MMRp                            210
Treatment Naive vs Anti PD1             210
T-cell exhaustion                       210
IO Response                             210
Treatment Naive vs Neoadjuvant Chemo    210
ER+ vs TNBC                             210
Treatment Naive vs TKI treated          168
Name: count, dtype: int64

In [22]:
calssification_metrics.exp.isna().sum()

np.int64(0)

In [23]:
calssification_metrics.to_csv('./results/classification_metrcis.csv')

In [24]:
calssification_metrics[['model', 'exp']]

Metrics,model,exp
0,HVG,Treatment Naive vs TKI treated
1,HVG,Treatment Naive vs TKI treated
2,HVG,Treatment Naive vs TKI treated
3,HVG,Treatment Naive vs TKI treated
4,HVG,MMRd vs MMRp
...,...,...
475,Nicheformer,Treatment Naive vs Neoadjuvant Chemo
476,Nicheformer,Treatment Naive vs Neoadjuvant Chemo
477,Nicheformer,Treatment Naive vs Neoadjuvant Chemo
478,Nicheformer,Treatment Naive vs Neoadjuvant Chemo


In [25]:
calssification_metrics.model.value_counts()

model
HVG               102
STATE             102
scFoundation      102
GF-V2 [cancer]    102
GF-V2-Deep        102
GF-V1             102
GF-V2             102
scGPT [cancer]    102
scGPT             102
CellPLM           102
SCimiarity        102
PCA [100]         102
scVI              102
Nicheformer       102
Name: count, dtype: int64